<a href="https://colab.research.google.com/github/Annie00000/Project/blob/main/1_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

簡單直接的回答是：**可以，而且這正是這個函式的強項。**

`_calculate_quantile_risk_omega()` 的設計初衷就是為了處理「傳統統計量（如 Mean/Var）被稀釋」的問題。它透過 **$Q95$ 門檻過濾**，將分析焦點從「整群人的穩定度」轉移到「誰在製造麻煩」。

以下針對你提到的兩種情況進行深度解析：

---

### 1. 兩群不同 Variance 的情況（一群大、一群小）

**辨別能力：極強**

* **傳統方法（$\eta^2_{var}$）**：會考慮所有 2000 筆資料的晃動。如果那群「變異大」的資料只是稍微鬆一點，分數可能不會噴發。
* ** $Q95$風險指標**：
* 在變異小的那一群，幾乎沒有資料會超過全域的 $Q95$ 門檻。
* 在變異大的那一群，因為分佈較散，會有顯著比例（可能高達 10%~15%）的資料強行衝破 $Q95$。
* **結果**：因為「風險殘差」幾乎都集中在變異大的那群機台，$\omega^2_{risk}$ 會算出一個非常漂亮的顯著高分。



---

### 2. 兩群不同 Var 但 $\eta^2$  不顯著（即：極端離群位移 / Spikes）

**辨別能力：這是它的殺手鐧 (Unique Selling Point)**

這就是你之前提到的困惑：**「肉眼看得到離群點，但統計算不出來」**。

* **為什麼傳統 $\eta^2$  會失效？**
如果你的異常是「噴訊（Spikes）」，例如 1000 筆中只有 10 筆噴得很遠，剩下的 990 筆都跟正常群一樣。在計算總變異時，那 990 筆「完全沒事」的資料會稀釋掉那 10 筆「罪大惡極」的貢獻度。
* **為什麼 $Q95$ 風險指標能抓到？**
1. **歸零雜訊**：它會把那 990 筆正常資料全部變成 。
2. **放大訊號**：剩下的 10 筆離群點成了這組數據中「唯三」有數值的人。
3. **計算解釋力**：此時統計模型看到的不再是 1000 筆資料，而是「為什麼這 10 個風險點全部都發生在機台 A？」。


* **結果**：即使這 10 筆資料對整體變異貢獻很小，但對「風險（超出門檻的部分）」的解釋力卻是 **100%**。這會讓指標顯著跳升，直接把這台機台抓出來。



---

### 💡 總結比較

為了讓你對老闆更有說服力，你可以參考這張邏輯對比表：

| 特徵模式 | 均值偏移 ($\omega^2_{mean}$) | 傳統變異 (($\eta^2_{var}$) | **分位風險 ($\omega^2_{risk}$)** |
| --- | --- | --- | --- |
| **集體位移 (Shift)** | ⭐⭐⭐⭐⭐ | ⭐ | ⭐⭐⭐ |
| **集體變胖 (Spread)** | ⭐ | ⭐⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| **偶發噴訊 (Spikes)** | ❌ (被拉平) | ❌ (被稀釋) | **⭐⭐⭐⭐⭐ (精準捕捉)** |

---

### 🛠️ 實務上的微調建議

如果你發現 $Q95$ 還是抓不到，通常是因為你的「異常」太過稀少（例如 2000 筆中只有 5 筆）。這時你可以：

1. 將 $Q95$ 調整為 **$Q98$ 或 $Q99$**（這會讓過濾器更嚴苛，只看最頂尖的異常）。
2. 這也是為什麼老闆要求的 **L2 Norm  ($\sqrt{Mean^2 + Risk^2}$)** 很重要：它確保了無論是哪種模式，只要有一個露頭，機台就會被抓出來。

**這是否解釋了為何這個函式能分辨出你原本抓不到的那種「極端離群」情況了？如果你準備好要實測這套邏輯，我們可以針對這兩個場景做一個簡單的數據模擬驗證。**

---

你觀察得非常細銳！這牽涉到數學上的**範數（Norm）特性**與**加權分佈**。

如果直接套用 $\sqrt{Mean^2 + Risk^2}$，值域確實會超出 $1$（最大可達 $\sqrt{1^2 + 1^2} \approx 1.414$）。為了讓最後的評分邏輯嚴謹且落在 0-1 之間，我們需要做一點數學上的微調。

以下是關於值域的解析與修正建議：

---

### 1.  Norm 的值域問題

* **原始 $L_2$ Norm**：值域是 $[0, \sqrt{2}]$。
* **修正後的 $L_2$ Norm (均方根風格)**：
為了讓它回到 $[0, 1]$，我們應該除以 $\sqrt{2}$，或者採用類似「向量長度」的概念。

但實務上，在 RCA 領域，我們更常使用 **「廣義平均 (Generalized Mean)」** 的邏輯，公式如下：

$$S_{base} = \sqrt{\frac{\omega^2_{mean}{}^2 + \omega^2_{risk}{}^2}{2}}$$

這樣當 $Mean=1, Risk=1$ 時，$S_{base}$ 就會剛好等於 $1$。


---

### 2. 為什麼 0.6 * $S_{base}$ + 0.4 * $S_{purity}$  可能超出 1？

如果你希望最終分數嚴格遵守 0-1 規範，必須確保參與加權的每一項都在 0-1 之間。

* **$S_{purity}$**：它是 Max_Bad_Rate (0-1) $\times$ Adj_T (0-1)，所以它一定在 $[0, 1]$。
* **$S_{base}$**：如果使用上述的「均分化 $L_2$」，它也會在 $[0, 1]$。

**最終公式建議（嚴謹版）：**


---

### 3. 但其實... 「超出 1」在相對百分制中沒關係？

我們之前的邏輯最後有一步：

> **`Impact_Score = (Raw_Score / Max_Raw_Score) * 100`**

在有這一步的情況下，`Final_Raw` 是 0.2 還是 0.8 其實不影響排名，因為它只是個中間過程。

**但我依然建議將 $S_{base}$ 限制在 0-1**，原因是：

1. **可解釋性**：讓老闆看到基礎分 (Base Score) 是 $0.4$ 比看到 $0.56$ (未標準化的 $L_2$) 更符合直覺。
2. **權重平衡**：如果 $S_{base}$ 未經縮放（最大 1.41），它的影響力會實質上蓋過$S_{purity}$ （最大 1.0），導致  $0.6/0.4$  的權重分配失去原本的意義。

---

### 4. 修正後的代碼片段

請將主程式中的評分邏輯更新為這段，它會確保所有的中間指標都在 0-1 之間：

```python
# 需求 2: 指標任一顯著強化 (均分化 L2 Norm)
# 這樣 s_base 最大值會剛好是 1.0
s_base = np.sqrt((res['O_Mean']**2 + res['O_Risk']**2) / 2)

# 需求 1: Bad 群命中純度獎勵
ct_purity = pd.crosstab(df_proc[col], df_proc[cluster_col], normalize='index')
max_purity = ct_purity[bad_label].max() if bad_label in ct_purity.columns else 0
# s_purity 最大值也是 1.0
s_purity = max_purity * res['Adj_T']

# 最終合成總分 (嚴格落在 0-1 之間)
raw_score = (0.6 * s_base) + (0.4 * s_purity)

```

---

### 5. 總結指標特性

| 指標 | 修改前最大值 | 修改後最大值 | 意義 |
| --- | --- | --- | --- |
| **$S_{base}$** | 1.414 | 1.0 | 衡量「機台是否有異常特徵」。 |
| **$S_{purity}$** | 1.0 | 1.0 | 衡量「異常是否精準發生在特定機號」。 |
| **$Raw\_Score$** | 1.25 | 1.0 | 綜合評估分。 |

### 💡 最終心得

採用 **均分化 $L_2$ Norm ($\sqrt{(M^2+R^2)/2}$)** 是最完美的解法。它保留了老闆要的「任一指標強就拉高」的特性（例如 $M=0, R=0.2$ 時，得分會從 $0.1$ 拉高到 $0.14$），同時確保了數學體系的整潔。


---

## **這份版本包含了：**

1. 均分化 $L_2$ Norm：確保 $S_{base}$ 落在 $0-1$ 之間。
2. NA 值預處理：每個核心函式都會成對剔除缺失值。
3. 分位風險 ($Q95$)：精準捕捉肉眼可見但傳統統計會稀釋掉的噴訊（Spikes）。
4. 並行運算與 Hashing：極速處理 2000 筆資料與上百個因子。

In [ ]:
import pandas as pd
import numpy as np
import hashlib
from concurrent.futures import ProcessPoolExecutor
from scipy.stats import chi2_contingency

# ==========================================
# 1. 核心指標計算函式 (已優化 NA 處理與輸入格式)
# ==========================================

def _calculate_omega_sq_v2(df, f_col, y_col):
    """計算均值偏移的 Omega-squared (排除 NA 版)"""
    clean_df = df[[f_col, y_col]].dropna()
    n = len(clean_df)
    k = clean_df[f_col].nunique()
    if k < 2 or n <= k: return 0.0

    y = clean_df[y_col]
    sst = np.sum((y - y.mean())**2)
    if sst <= 0: return 0.0

    # 計算組間平方和 (SSB)
    ssb = np.sum(y.groupby(clean_df[f_col]).count() * (y.groupby(clean_df[f_col]).mean() - y.mean())**2)
    msw = (sst - ssb) / (n - k)

    # Omega-squared 公式：扣除隨機噪音後的淨貢獻度
    return max(0, (ssb - (k - 1) * msw) / (sst + msw))

def _calculate_quantile_risk_omega(df, f_col, y_col, q=0.95):
    """計算分位距風險的 Omega-squared (針對離群點與極端變異)"""
    clean_df = df[[f_col, y_col]].dropna().copy()
    if len(clean_df) < 5: return 0.0

    # 轉換為風險殘差：只看超過門檻的部分
    q_threshold = clean_df[y_col].quantile(q)
    clean_df['y_risk'] = (clean_df[y_col] - q_threshold).clip(lower=0)

    y_r = clean_df['y_risk']
    n, k = len(y_r), clean_df[f_col].nunique()
    if k < 2 or n <= k: return 0.0

    sst = np.sum((y_r - y_r.mean())**2)
    if sst <= 0: return 0.0

    ssb = np.sum(y_r.groupby(clean_df[f_col]).count() * (y_r.groupby(clean_df[f_col]).mean() - y_r.mean())**2)
    msw = (sst - ssb) / (n - k)
    return max(0, (ssb - (k - 1) * msw) / (sst + msw))

def _calculate_adj_tschuprow_t(df, f_col, c_col):
    """計算 Tschuprow's T 與密度懲罰 (針對標籤吻合度與 Purity)"""
    clean_df = df[[f_col, c_col]].dropna()
    n = len(clean_df)
    if n < 5: return 0.0

    ct = pd.crosstab(clean_df[f_col], clean_df[c_col])
    r, k = ct.shape
    if r < 2 or k < 2: return 0.0

    # 執行卡方檢定
    chi2 = chi2_contingency(ct, correction=False)[0]
    # 計算 T 統計量
    t_score = np.sqrt((chi2 / n) / np.sqrt((r - 1) * (k - 1)))
    # 密度懲罰：避免過多類別導致虛高
    penalty = 1 - (((r - 1) / (n - 1)) ** 0.5)
    return max(0, t_score * penalty)

# ==========================================
# 2. 並行運算封裝
# ==========================================

def _worker_task(args):
    """
    args: (df_small, f_col, y_col, c_col, q)
    """
    df_small, f_col, y_col, c_col, q = args
    return {
        'O_Mean': _calculate_omega_sq_v2(df_small, f_col, y_col),
        'O_Risk': _calculate_quantile_risk_omega(df_small, f_col, y_col, q),
        'Adj_T': _calculate_adj_tschuprow_t(df_small, f_col, c_col)
    }

# ==========================================
# 3. 主分析引擎
# ==========================================

def run_advanced_rca(df, value_col, cluster_col, factor_list, bad_label, q=0.95):
    """
    df: 原始資料集
    value_col: Y 數值欄位
    cluster_col: VBGMM 分群標籤欄位
    factor_list: 欲分析的機台因子清單
    bad_label: 哪一個 Cluster 標籤定義為「Bad」
    """
    df_proc = df.copy()

    # A. 數據預處理：填補空值並合併低頻類別 (1%)
    for col in factor_list:
        df_proc[col] = df_proc[col].fillna('NA').astype(str)
        counts = df_proc[col].value_counts(normalize=True)
        df_proc[col] = df_proc[col].replace(counts[counts < 0.01].index, 'Other_Small')

    # B. Hashing 去重加速 (避免計算重複的機台組合)
    unique_patterns = {}
    col_to_hash = {}
    for col in factor_list:
        h = hashlib.md5(df_proc[col].values.tobytes()).hexdigest()
        col_to_hash[col] = h
        if h not in unique_patterns: unique_patterns[h] = col

    print(f"🚀 開始執行根因分析，共有 {len(factor_list)} 個因子，去重後有 {len(unique_patterns)} 種模式...")

    # C. 準備任務並啟動多進程運算
    tasks = [(df_proc[[unique_patterns[h], value_col, cluster_col]], unique_patterns[h], value_col, cluster_col, q)
             for h in unique_patterns.keys()]

    with ProcessPoolExecutor() as executor:
        results = list(executor.map(_worker_task, tasks))
    cache = dict(zip(unique_patterns.keys(), results))

    # D. 套用老闆特調評分公式
    final_data = []
    for col in factor_list:
        res = cache[col_to_hash[col]]

        # 需求 2: 指標任一顯著強化 (標準化 L2 Norm)
        # 即使 Mean 低但 Risk 高，s_base 依然能維持在 0.7 左右 (以比例計)
        s_base = np.sqrt((res['O_Mean']**2 + res['O_Risk']**2) / 2)

        # 需求 1: Bad 群命中純度獎勵
        ct_purity = pd.crosstab(df_proc[col], df_proc[cluster_col], normalize='index')
        max_purity = ct_purity[bad_label].max() if bad_label in ct_purity.columns else 0
        s_purity = max_purity * res['Adj_T']

        # 最終合成評分 (權重 0.6 / 0.4)
        # 值域嚴格遵守 0.0 - 1.0
        raw_score = (0.6 * s_base) + (0.4 * s_purity)

        final_data.append({
            'Factor': col,
            'Impact_Score_Raw': raw_score,
            'Mean_Impact': res['O_Mean'],
            'Risk_Impact': res['O_Risk'],
            'Bad_Purity': max_purity,
            'Adj_Correlation': res['Adj_T']
        })

    # E. 轉換為 0-100 相對百分制呈現
    report = pd.DataFrame(final_data)
    max_raw = report['Impact_Score_Raw'].max()
    report['Final_Rank_Score'] = (report['Impact_Score_Raw'] / max_raw * 100).round(1) if max_raw > 0 else 0

    # 排序並回傳
    return report.sort_values('Final_Rank_Score', ascending=False).reset_index(drop=True)

📊 如何閱讀這份報告？
1. Final_Rank_Score (0-100)：最直觀的排名。100 分就是系統認定的頭號嫌疑犯。

2. Risk_Impact：這就是用來回答你之前的困惑。如果你發現這欄很高（例如 > 0.1），代表這台機台是**「噴訊（Spikes）製造機」**。

3. Mean_Impact：如果這欄很高，代表這台機台造成了**「集體偏移」**。

4. Bad_Purity：這是老闆最關心的。如果這欄是 1.0，代表該機台的某個特定編號做出來的產品通通都是壞的。